# Notebook 06 — Evaluation And Paper Results



**Goal:** Aggregate the final ML results for the event-safety version of CrowdVision.



This notebook focuses on:



1. Final density estimation evaluation

2. Final congestion forecasting evaluation

3. Final anomaly detection evaluation

4. Multi-task ablation review

5. Event-safety dataset gap analysis

6. Paper-ready figures and exports



> The scope here is event safety intelligence, not person identity tracking.

In [ ]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys

import json



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))



import torch

import numpy as np

import matplotlib.pyplot as plt

import matplotlib



# Publication-ready matplotlib settings

matplotlib.rcParams.update({

    'font.size'     : 11,

    'axes.labelsize': 12,

    'legend.fontsize': 10,

    'figure.dpi'    : 150,

})



from src.data_loaders import (

    get_shanghaitech_loaders, get_jhu_loaders,

    load_metr_la, get_ucsd_loaders, get_market1501_loaders

)

from src.models import CSRNet, AdaptiveCSRNet, GCNGRU, AdaptiveNASGNN, ConvAE, FutureFrameNet

from src.models.multitask.unified import UnifiedCrowdVision

from src.evaluation import evaluate_density, evaluate_forecasting, evaluate_anomaly_detection

from src.utils import make_results_table



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 2: Load all checkpoints ─────────────────────────────────────────────
def load_model(model_cls, ckpt_path, device, **model_kwargs):
    """Load a saved checkpoint into a model. Returns None if checkpoint missing."""
    path = Path(ckpt_path)
    if not path.exists():
        print(f'  [MISSING] {path.name}  →  {path.parent.name}')
        return None
    model = model_cls(**model_kwargs).to(device)
    ckpt  = torch.load(path, map_location=device)
    state = ckpt.get('model', ckpt)
    model.load_state_dict(state, strict=False)
    model.eval()
    print(f'  [LOADED]  {path.parent.name}/{path.name}')
    return model

print('Loading models from checkpoints...')

m_csrnet_A      = load_model(CSRNet,          CKPT_ROOT / 'csrnet_shaA'          / 'best.pt', DEVICE, load_weights=False)
m_adaptive_A    = load_model(AdaptiveCSRNet,   CKPT_ROOT / 'adaptive_csrnet_shaA' / 'best.pt', DEVICE, load_weights=False)
m_csrnet_B      = load_model(CSRNet,          CKPT_ROOT / 'csrnet_shaB'          / 'best.pt', DEVICE, load_weights=False)
m_adaptive_B    = load_model(AdaptiveCSRNet,   CKPT_ROOT / 'adaptive_csrnet_shaB' / 'best.pt', DEVICE, load_weights=False)
m_convae        = load_model(ConvAE,           CKPT_ROOT / 'convae_ped2'          / 'best.pt', DEVICE, in_channels=1, base_ch=32)
m_ffnet         = load_model(FutureFrameNet,   CKPT_ROOT / 'ffnet_ped2'           / 'best.pt', DEVICE, num_input_frames=7, in_channels=1)

print('\nCheckpoints status review done.')

In [ ]:
# ── Cell 3: Full density evaluation (SHA-A, SHA-B) ────────────────────────────
density_results = {}

for part, m_cs, m_ad in [
    ('A', m_csrnet_A, m_adaptive_A),
    ('B', m_csrnet_B, m_adaptive_B),
]:
    print(f'\nLoading ShanghaiTech Part {part}...')
    _, test_l = get_shanghaitech_loaders(
        data_root=str(DATA_ROOT), part=part, batch_size=1, target_size=(576, 768)
    )

    if m_cs:
        res = evaluate_density(m_cs, test_l, DEVICE)
        density_results[f'CSRNet SHA-{part}']         = res
        print(f'  CSRNet SHA-{part}       MAE={res["mae"]:.2f}  MSE={res["mse"]:.2f}')
    if m_ad:
        res = evaluate_density(m_ad, test_l, DEVICE)
        density_results[f'AdaptiveCSRNet SHA-{part}'] = res
        print(f'  AdaptiveCSRNet SHA-{part} MAE={res["mae"]:.2f}  MSE={res["mse"]:.2f}')

In [ ]:
# ── Cell 4: SOTA comparison table — Density Estimation ───────────────────────
#
# Published results from papers (add/update these as you find them)
SOTA_DENSITY = {
    # Model: (SHA-A MAE, SHA-A MSE, SHA-B MAE, SHA-B MSE)
    'MCNN (2016)':        (110.2, 173.2, 26.4, 41.3),
    'CSRNet (2018)':      ( 68.2, 115.0,  10.6, 16.0),
    'SFCN (2019)':        ( 64.8, 107.5,   7.6, 13.0),
    'BL (2019)':          ( 62.8, 101.8,   7.7, 12.7),
    'DM-Count (2020)':    ( 59.7,  95.7,   7.4, 11.8),
    'SASNet (2021)':      ( 53.6,  88.4,   6.1,  9.9),
    'P2PNet (2021)':      ( 52.7,  85.1,   6.2, 10.0),
    'CLTR (2022)':        ( 50.3,  82.0,   5.9,  9.4),
}

print('\n=== Density Estimation SOTA Comparison ===')
print(f'{"Model":<30} {"SHA-A MAE":>10} {"SHA-A MSE":>10} {"SHA-B MAE":>10} {"SHA-B MSE":>10}')
print('-' * 70)

for model_name, (mae_a, mse_a, mae_b, mse_b) in SOTA_DENSITY.items():
    print(f'{model_name:<30} {mae_a:>10.1f} {mse_a:>10.1f} {mae_b:>10.1f} {mse_b:>10.1f}')

print('-' * 70)
for key, res in density_results.items():
    if 'SHA-A' in key:
        res_b = density_results.get(key.replace('SHA-A', 'SHA-B'), {})
        print(f'{key.replace(" SHA-A", " (ours)"):<30} '
              f'{res.get("mae", 0):>10.2f} {res.get("mse", 0):>10.2f} '
              f'{res_b.get("mae", 0):>10.2f} {res_b.get("mse", 0):>10.2f}')

In [ ]:
# ── Cell 5: Full forecasting evaluation ──────────────────────────────────────
forecasting_results = {}

metr_h5  = DATA_ROOT / 'metr-la' / 'metr-la.h5'
metr_pkl = DATA_ROOT / 'metr-la' / 'adj_mx.pkl'

if metr_h5.exists():
    print('Loading METR-LA for final evaluation...')
    _, _, test_l, scaler, adj, num_nodes = load_metr_la(
        h5_path=str(metr_h5),
        adj_path=str(metr_pkl) if metr_pkl.exists() else None,
        seq_in=12, seq_out=12, batch_size=64,
    )

    gcn_ckpt = CKPT_ROOT / 'gcn_gru_metrla' / 'best.pt'
    nas_ckpt = CKPT_ROOT / 'nas_gnn_retrain' / 'best.pt'

    for name, ckpt_path, model_cls in [
        ('GCN-GRU', gcn_ckpt, GCNGRU),
        ('AdaptiveNAS-GNN', nas_ckpt, AdaptiveNASGNN),
    ]:
        if not ckpt_path.exists():
            print(f'  {name}: checkpoint missing at {ckpt_path}')
            continue
        m = model_cls(num_nodes=num_nodes, in_features=2, hidden_dim=64,
                      num_layers=2, seq_out=12).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        m.load_state_dict(ckpt.get('model', ckpt), strict=False)
        m.eval()
        res = evaluate_forecasting(m, test_l, scaler, adj, DEVICE, horizons=[3, 6, 12])
        forecasting_results[name] = res
        print(f'  {name} @60min → MAE: {res.get("h12", {}).get("mae", float("nan")):.4f}')
else:
    print('METR-LA data not found, skipping forecasting evaluation.')

In [ ]:
# ── Cell 6: SOTA comparison — Traffic Forecasting ─────────────────────────────
SOTA_FORECASTING = {
    #                     15min          30min          60min
    #                  MAE  RMSE MAPE  MAE  RMSE MAPE  MAE  RMSE MAPE
    'DCRNN (2018)'    : [(2.77,5.38,7.30),(3.15,6.45,8.80),(3.60,7.59,10.50)],
    'STGCN (2018)'    : [(2.88,5.74,7.62),(3.47,7.24,9.57),(4.59,9.40,12.70)],
    'GWNET (2019)'    : [(2.69,5.15,6.90),(3.07,6.22,8.37),(3.53,7.37,10.01)],
    'AGCRN (2020)'    : [(2.87,5.58,7.41),(3.23,6.58,8.98),(3.62,7.51,10.38)],
    'MTGNN (2020)'    : [(2.69,5.18,6.86),(3.05,6.17,8.19),(3.49,7.23, 9.87)],
    'STID (2022)'     : [(2.65,5.07,6.73),(3.01,6.07,8.12),(3.48,7.19, 9.82)],
}

horizons = ['15 min', '30 min', '60 min']

print('\n=== METR-LA Traffic Forecasting SOTA ===')
header = f'{"Model":<24}'
for h in horizons:
    header += f' {h+" MAE":>10} {h+" RMSE":>11} {h+" MAPE%":>11}'
print(header)
print('-' * (24 + 33*3))

for model_name, row in SOTA_FORECASTING.items():
    line = f'{model_name:<24}'
    for mae, rmse, mape in row:
        line += f' {mae:>10.2f} {rmse:>11.2f} {mape:>11.2f}'
    print(line)

print('-' * (24 + 33*3))
horizon_keys = ['h3', 'h6', 'h12']
for model_name, res in forecasting_results.items():
    line = f'{model_name+" (ours)":<24}'
    for hk in horizon_keys:
        h_data = res.get(hk, {})
        mae  = h_data.get('mae',  float('nan'))
        rmse = h_data.get('rmse', float('nan'))
        mape = h_data.get('mape', float('nan'))
        line += f' {mae:>10.2f} {rmse:>11.2f} {mape:>11.2f}'
    print(line)

In [ ]:
# ── Cell 7: Full anomaly detection evaluation ─────────────────────────────────
anomaly_results = {}

for ped in ['ped1', 'ped2']:
    train_l, test_l = get_ucsd_loaders(
        data_root=str(DATA_ROOT), ped=ped,
        img_size=(128, 192), batch_size=16, clip_len=8,
    )

    for name, ckpt_path, cls, cls_kwargs in [
        ('ConvAE',        CKPT_ROOT / f'convae_{ped}'  / 'best.pt', ConvAE,       {'in_channels':1,'base_ch':32}),
        ('FutureFrameNet',CKPT_ROOT / f'ffnet_{ped}'   / 'best.pt', FutureFrameNet,{'num_input_frames':7,'in_channels':1,'base_ch':32}),
    ]:
        if not Path(ckpt_path).exists():
            print(f'  [{ped}] {name}: checkpoint missing')
            continue
        m = cls(**cls_kwargs).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        m.load_state_dict(ckpt.get('model', ckpt), strict=False)
        m.eval()
        res = evaluate_anomaly_detection(m, train_l, test_l, DEVICE)
        anomaly_results[f'{name} {ped.upper()}'] = res
        print(f'  {name} {ped.upper()} → AUC: {res["auc"]:.2f}%  EER: {res["eer"]:.2f}%')

In [ ]:
# ── Cell 8: SOTA comparison — Anomaly Detection ───────────────────────────────
# Frame-level AUC on UCSD Ped2 (from published papers)
SOTA_ANOMALY_PED2 = {
    'Conv-AE (2016)':      81.0,
    'DeepOC (2019)':       83.7,
    'Future Frame (2018)': 95.4,
    'CLOZE (2021)':        97.3,
    'EVAL (2022)':         97.6,
}

print('\n=== Anomaly Detection AUC — UCSD Ped2 ===')
print(f'{"Model":<35} {"AUC (%)"}');  print('-'*45)

for name, auc in SOTA_ANOMALY_PED2.items():
    print(f'{name:<35} {auc:.1f}')

print('-'*45)
for key, res in anomaly_results.items():
    if 'PED2' in key:
        print(f'{key+" (ours)":<35} {res["auc"]:.1f}')

In [ ]:
# ── Cell 9: Ablation study table ─────────────────────────────────────────────
ablation_md_path = EXPS_ROOT / 'multitask_ablation.md'

ablation_configs = [
    # (description, config changes)
    ('Full model',              'all components'),
    ('w/o CBAM attention',      'remove CBAM from AdaptiveCSRNet'),
    ('w/o multi-scale backend', 'single-scale 3×3 convs only'),
    ('w/o SSIM loss',           'MSE loss only'),
    ('w/o consistency loss',    'no cross-task regularisation'),
]

print('=== Ablation Study Design ===')
print('\nTo reproduce each ablation, change the model/config and re-run:')
print(f'{"Config":<35} {"Change"}')
print('-'*75)
for name, change in ablation_configs:
    print(f'{name:<35} {change}')

print()
if ablation_md_path.exists():
    print('Automated ablation (from Notebook 05):')
    print(ablation_md_path.read_text())
else:
    print('Run Notebook 05 to generate automated ablation results.')

In [ ]:
# ── Cell 10: Publication figure — density comparison bar chart ────────────────
models_a = ['MCNN','CSRNet','SFCN','BL','DM-Count','SASNet','P2PNet','CLTR']
mae_a    = [110.2,  68.2,  64.8, 62.8,  59.7,  53.6,  52.7,  50.3]
mse_a    = [173.2, 115.0, 107.5,101.8,  95.7,  88.4,  85.1,  82.0]

# Add our models if available
if 'CSRNet SHA-A' in density_results:
    models_a.append('CSRNet (ours)')
    mae_a.append(density_results['CSRNet SHA-A']['mae'])
    mse_a.append(density_results['CSRNet SHA-A']['mse'])
if 'AdaptiveCSRNet SHA-A' in density_results:
    models_a.append('AdaptiveCSRNet (ours)')
    mae_a.append(density_results['AdaptiveCSRNet SHA-A']['mae'])
    mse_a.append(density_results['AdaptiveCSRNet SHA-A']['mse'])

colors = ['#4C72B0']*len(SOTA_DENSITY) + ['#DD8452', '#C44E52']
x = np.arange(len(models_a))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Crowd Density Estimation — ShanghaiTech Part A', fontsize=13)

for ax, vals, ylabel in zip(axes, [mae_a, mse_a], ['MAE', 'MSE']):
    bars = ax.bar(x, vals, color=colors[:len(vals)], zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(models_a, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis='y', alpha=0.3, zorder=0)
    ax.set_title(f'{ylabel} (lower is better)')
    # annotate ours
    for i, (b, v) in enumerate(zip(bars, vals)):
        if i >= len(SOTA_DENSITY):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                    f'{v:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='darkred')

plt.tight_layout()
plt.savefig(EXPS_ROOT / 'paper_density_comparison.pdf', bbox_inches='tight')
plt.savefig(EXPS_ROOT / 'paper_density_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved paper figure.')

In [ ]:
# Dataset gap analysis for the event-safety semester scope

print('=' * 70)

print('DATASET SUFFICIENCY ANALYSIS FOR EVENT-SAFETY CROWDVISION')

print('=' * 70)



have_datasets = {

    'ShanghaiTech A/B': DATA_ROOT / 'shanghaitech_part_A_final',

    'JHU-Crowd++': DATA_ROOT / 'jhu_crowd_v2.0',

    'UCSD Ped1/Ped2': DATA_ROOT / 'UCSD_Anomaly_Dataset',

    'METR-LA': DATA_ROOT / 'metr-la' / 'metr-la.h5',

    'PEMS-D3/D4/D7': DATA_ROOT / 'metr-la',

}



recommended_additions = {

    'UCF-QNRF': 'Best single extra density benchmark to strengthen paper credibility.',

    'NWPU-Crowd': 'Largest modern crowd counting benchmark; strongest optional density upgrade.',

    'PEMS-BAY': 'Best second benchmark for congestion forecasting.',

    'ShanghaiTech Campus': 'Useful anomaly benchmark closer to event-safety scenes.',

}



not_needed_now = {

    'MOT17 / MOT20': 'Not required because this semester scope does not center on person identity tracking.',

    'CrowdHuman': 'Optional only if detection-heavy pipeline becomes necessary later.',

    'DukeMTMC': 'Out of scope for current event-safety narrative.',

}



print('\nAVAILABLE NOW:')

for name, path in have_datasets.items():

    status = '✓ found' if Path(path).exists() else '✗ missing'

    print(f'  {name:<25} {status}')



print('\nHIGH-VALUE ADDITIONS:')

for name, reason in recommended_additions.items():

    print(f'  {name:<20} {reason}')



print('\nNOT REQUIRED FOR THIS SEMESTER VERSION:')

for name, reason in not_needed_now.items():

    print(f'  {name:<20} {reason}')

In [ ]:
# Final event-safety readiness summary

readiness = {

    'density_ready': bool(density_results),

    'forecasting_ready': bool(forecasting_results),

    'anomaly_ready': bool(anomaly_results),

    'paper_scope': 'event_safety_ml_core',

    'recommended_next_downloads': ['UCF-QNRF', 'PEMS-BAY'],

    'not_required_this_semester': ['MOT17', 'CrowdHuman', 'DukeMTMC'],

}



readiness_path = EXPS_ROOT / 'event_safety_readiness.json'

with open(readiness_path, 'w') as f:

    json.dump(readiness, f, indent=2)



print(json.dumps(readiness, indent=2))

print(f'Saved readiness summary to {readiness_path}')

In [ ]:
# ── Cell 13: Final consolidated results export ────────────────────────────────
all_results_json = {
    'density': {k: {m: float(v) for m, v in r.items()} for k, r in density_results.items()},
    'anomaly': {k: {m: float(v) for m, v in r.items()} for k, r in anomaly_results.items()},
    'forecasting': {
        k: {h: {m: float(v) for m, v in hv.items()} for h, hv in r.items()}
        for k, r in forecasting_results.items()
    },
}

json_path = EXPS_ROOT / 'all_results.json'
with open(json_path, 'w') as f:
    json.dump(all_results_json, f, indent=2)

print(f'All results saved to {json_path}')
print('\nTotal results stored:')
for task, data in all_results_json.items():
    print(f'  {task}: {len(data)} model-dataset combinations')

---



## Evaluation Complete



### Main outputs saved in `experiments/`



- `all_results.json`

- `paper_density_comparison.pdf`

- `paper_density_comparison.png`

- `event_safety_readiness.json`

- density, forecasting, anomaly, and multitask result tables



### Practical next steps



1. Run the notebooks in order and let checkpoints accumulate.

2. If one extra dataset can be added, add UCF-QNRF first.

3. If a second extra dataset can be added, add PEMS-BAY.

4. Hand the generated risk summaries, CSVs, and figures to the frontend and backend team.



This is enough for the semester event-safety ML core without expanding into person identification.